# TEP dengan DAIL-SQL sebagai B — GraphRAG 1-Stage dan 2-Stage, per model

Notebook ini menghitung **TEP (Token Elasticity of Performance)** dari hasil run yang sudah disimpan, jadi **tidak butuh GPU** dan tidak menjalankan ulang pipeline. Logikanya ada di `src/experiments/tep_steps.py`.

$$\mathrm{TEP}_c = \frac{\Delta EX_c / EX_B}{\Delta T_c / T_B}, \qquad \Delta EX_c = EX_c - EX_B, \quad \Delta T_c = T_c - T_B$$

- **B = DAIL-SQL**, baseline skripsi (poin 38, 43). Sebelumnya B = GraphRAG 1-Stage; TEP itu sudah dihapus dari `eval_pipeline.ipynb` (poin 45).
- **c = GraphRAG 1-Stage** dan **c = GraphRAG 2-Stage**, masing-masing terhadap DAIL-SQL, untuk tiap model (3B, 7B, 14B).
- T = T_in + α × T_out per pertanyaan, lalu dirata-rata. Rumus dan fungsi sama dengan `src/metrics/tep.py` (`compute_tep()`), tidak diubah.

## File yang dibutuhkan

| Kondisi | Dari notebook | Taruh di |
|---|---|---|
| 1-Stage dan 2-Stage | `eval_pipeline.ipynb` → `hasil_<MODEL>.zip` | ekstrak ke `outputs/models/` (isinya folder `Qwen2.5-Coder-<MODEL>-Instruct/`) |
| DAIL-SQL | `eval_dail.ipynb` → `hasil_dail_<MODEL>.zip` | ekstrak ke `outputs/models/Qwen2.5-Coder-<MODEL>-Instruct/` (isinya `tables/dail/`, `raw_logs/`) |
| Alternatif tanpa zip | `hasil_skripsi_<MODEL>.xlsx`, `hasil_dail_<MODEL>.xlsx` | `outputs/tables/` |

Zip lebih baik daripada Excel: metadata run (nama model lengkap, `SAMPLE_RATIO`, k few-shot) ikut terbaca dan bisa dicek. Excel hanya dipakai kalau zip untuk model dan kondisi itu tidak ada; angka EX, T_in, T_out-nya sama (sheet `Per Query`).

## Pengecekan sebelum TEP dihitung

Baris yang tidak lolos tetap tampil dengan status `skipped` dan alasannya, tidak membuat notebook berhenti.

- **Model sama** (ukuran model, dan nama lengkap kalau metadatanya ada).
- **Pertanyaan sama persis** (`query_id`, database, gold SQL). Artinya `SAMPLE_RATIO` dan seed harus sama di `eval_pipeline` dan `eval_dail`.
- **DAIL-SQL dengan k = 9** (baseline skripsi). Run ablation `DAIL_K_SHOT` lain tidak dipakai.
- k few-shot 1-Stage/2-Stage memang berbeda dengan DAIL (tiap metode memakai best config-nya, poin 43), jadi hanya ditampilkan di kolom `Few-shot k` dan `DAIL k`.

## 1. Setup

In [ ]:
# os untuk pindah folder, subprocess untuk menjalankan perintah terminal
import os, subprocess
# Path untuk menulis lokasi file
from pathlib import Path

# Alamat dan branch repo, hanya dipakai kalau notebook jalan di luar repo (mis. Kaggle)
REPO_URL = "https://github.com/william-darma-wijaya/thesis-2803.git"
REPO_BRANCH = "main"

# Folder dianggap root repo kalau berisi file milik repo ini
def is_repo_root(p: Path) -> bool:
    return (p / "src" / "experiments" / "tep_steps.py").is_file()

# Path di notebook ini relatif terhadap root repo (outputs/models/, outputs/tables/)
if not is_repo_root(Path.cwd()):
    # Dijalankan dari notebooks/: naik satu folder
    if is_repo_root(Path.cwd().parent):
        os.chdir("..")
    # Di luar repo: clone dulu kalau belum ada
    else:
        if not is_repo_root(Path("thesis-2803").resolve()):
            subprocess.run(f"git clone -b {REPO_BRANCH} {REPO_URL}", shell=True, check=True)
        os.chdir("thesis-2803")
# Lokasi root repo
REPO_ROOT = Path.cwd()
print("Repo root:", REPO_ROOT)

## 2. Import dan Konfigurasi

In [ ]:
# sys untuk mendaftarkan lokasi kode
import sys
# Daftarkan root repo supaya modul src bisa di-import
sys.path.insert(0, str(REPO_ROOT))

# Konfigurasi utama: α dan k DAIL diambil dari sini, tidak ditulis ulang
from src.core.config import PipelineConfig
# Fungsi untuk menampilkan tabel di notebook
from src.experiments.result_tables import show
# Semua langkah notebook ini
from src.experiments import tep_steps as ts

# Konfigurasi bawaan
cfg = PipelineConfig()
# Bobot token output, sama untuk semua kondisi: T = T_in + α × T_out
ALPHA = cfg.token_output_weight
# k DAIL-SQL baseline skripsi (9)
DAIL_K = cfg.dail_k_shot
# Model yang diharapkan: 3B, 7B, 14B (yang belum ada hasilnya tetap muncul sebagai baris skipped)
MODELS = [ts.model_size(m) for m in cfg.llm_model_frame]
# Folder yang dicari: zip yang diekstrak dan Excel unduhan
INPUT_DIRS = [Path("outputs/models"), Path("outputs/tables")]
# Folder hasil notebook ini
OUT_DIR = Path("outputs/tables")

print(f"α = {ALPHA}, DAIL k = {DAIL_K}, model = {MODELS}")

## 3. Cari Hasil Run

Tabel di bawah menampilkan sumber yang dipakai per model dan kondisi. Periksa kolom `Queries`, `Sample ratio`, dan `Few-shot k` sebelum membaca TEP.

In [ ]:
# Cari hasil 1-Stage, 2-Stage, dan DAIL-SQL tiap model
runs, notes = ts.find_runs(INPUT_DIRS, DAIL_K)
# Tampilkan catatan (folder tidak ada, sumber ganda, Excel yang tidak dipakai)
for note in notes:
    print("[INFO]", note)
# Tampilkan sumber yang dipakai
show(ts.sources_table(runs))

## 4. TEP terhadap DAIL-SQL

Urutan kolom mengikuti guide Dimensi 1: angka mentah dulu (EX dan T kedua kondisi), baru TEP. ΔEX dalam poin persen.

| Nilai TEP | Interpretasi (guide Bagian 3, Dimensi 1) |
|---|---|
| TEP < 0 | Skenario ideal: GraphRAG meningkatkan EX **dan** menurunkan Token Consumption dibanding baseline |
| TEP ≈ 0 (±0.05) | Efisiensi token tercapai tanpa trade-off penurunan EX yang signifikan |
| TEP > 0 | Ada trade-off: penurunan token disertai penurunan EX |

Tabel interpretasi itu mengasumsikan ΔT < 0. Kalau ΔT > 0 (pipeline memakai token lebih banyak dari DAIL-SQL), TEP > 0 berarti EX naik, jadi baca tanda ΔEX dan ΔT di kolomnya sebelum memakai tabel ini.

In [ ]:
# Satu baris per model × pipeline, B = DAIL-SQL
rows = ts.tep_rows(runs, ALPHA, DAIL_K, MODELS)
# Tampilkan tabel TEP
show(ts.tep_table(rows))

## 5. Simpan

`outputs/tables/hasil_tep.json` (angka mentah) dan `outputs/tables/hasil_tep.xlsx` (sheet `TEP` dan `Sources`).

In [ ]:
# Simpan JSON dan Excel
json_path, xlsx_path = ts.save_results(rows, runs, notes, ALPHA, DAIL_K, OUT_DIR)
print(f"Tersimpan: {json_path}\nTersimpan: {xlsx_path}")